<a href="https://colab.research.google.com/github/Alessandro-json/AI_PostProcessing_Detection/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2 - Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios

# Imports

In [21]:
import os
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# Globals


In [22]:
# Choose where the project will be stored in Colab.
WORKDIR = Path('/content')

REPO_URL = 'https://github.com/Alessandro-json/AI_PostProcessing_Detection'

# Repository folder name after git clone.
REPO_DIR = WORKDIR / 'REPO'

# Main paths used by the scripts.
TRAIN_CSV = 'data/splits/train_balanced.csv'
VAL_CSV = 'data/splits/val_balanced.csv'
TEST_CSV = 'data/splits/test_balanced.csv'

IMAGE_ROOT = "/content/data/raw/RRDataset_subset"
CHECKPOINT_NAME = 'best_rgb.pt'
CHECKPOINT_PATH = f'checkpoints/{CHECKPOINT_NAME}'
OUTPUT_DIR = 'results/rgb_baseline'
DEPTH_ROOT = "/content/drive/MyDrive/CV_Project/depth_maps"

DATASET_FILE_ID = "1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI"
DATASET_ZIP_PATH = "/content/RRDataset_subset.zip"

# Training hyperparameters for the first baseline.
EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 2
LEARNING_RATE = 1e-4

# Multi-task loss weights.
LAMBDA_FAKE = 1.0
LAMBDA_TRANSFORM = 1.0


# Setup repository



In [23]:
%cd {WORKDIR}

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


/content
/content/REPO
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 2.72 KiB | 697.00 KiB/s, done.
From https://github.com/Alessandro-json/AI_PostProcessing_Detection
   c27b596..aa3a644  main       -> origin/main
Updating c27b596..aa3a644
Fast-forward
 notebook.ipynb | 398 ++++++++++++++++++++++++++-------------------------------
 1 file changed, 184 insertions(+), 214 deletions(-)


# Install dependencies

In [24]:
# Install project dependencies.
!pip install -r requirements.txt


In [25]:
!pip install -q gdown

# Utils

In [26]:
def show_csv_summary(csv_path):
    """Print a quick summary of one split CSV."""
    path = Path(csv_path)
    if not path.exists():
        print(f'Missing file: {path}')
        return None

    df = pd.read_csv(path)
    print(f'File: {path}')
    print(f'Rows: {len(df)}')
    print('Columns:', list(df.columns))

    if 'fake_label' in df.columns:
        print('\nFake label distribution:')
        print(df['fake_label'].value_counts().sort_index())

    if 'transform_label' in df.columns:
        print('\nTransform label distribution:')
        print(df['transform_label'].value_counts().sort_index())

    if {'fake_label', 'transform_label'}.issubset(df.columns):
        print('\nJoint distribution:')
        print(pd.crosstab(df['transform_label'], df['fake_label'], rownames=['transform'], colnames=['fake']))

    return df


def show_image_exists_check(df, image_root, n=5):
    """Check whether the first n image paths exist."""
    if df is None:
        return

    root = Path(image_root)
    print(f'Image root: {root}')

    for rel_path in df['image_path'].head(n):
        full_path = root / rel_path
        print(full_path, 'OK' if full_path.exists() else 'MISSING')


# Data


In [27]:
train_df = show_csv_summary(TRAIN_CSV)


File: data/splits/train_balanced.csv
Rows: 2100
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    1050
1    1050
Name: count, dtype: int64

Transform label distribution:
transform_label
0    700
1    700
2    700
Name: count, dtype: int64

Joint distribution:
fake         0    1
transform          
0          350  350
1          350  350
2          350  350


In [28]:
val_df = show_csv_summary(VAL_CSV)


File: data/splits/val_balanced.csv
Rows: 450
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    225
1    225
Name: count, dtype: int64

Transform label distribution:
transform_label
0    150
1    150
2    150
Name: count, dtype: int64

Joint distribution:
fake        0   1
transform        
0          75  75
1          75  75
2          75  75


In [29]:
test_df = show_csv_summary(TEST_CSV)


File: data/splits/test_balanced.csv
Rows: 450
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    225
1    225
Name: count, dtype: int64

Transform label distribution:
transform_label
0    150
1    150
2    150
Name: count, dtype: int64

Joint distribution:
fake        0   1
transform        
0          75  75
1          75  75
2          75  75


In [30]:
!gdown --id "$DATASET_FILE_ID" -O "$DATASET_ZIP_PATH"

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI
From (redirected): https://drive.google.com/uc?id=1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI&confirm=t&uuid=d40edebd-a92e-4009-9034-e3359f385981
To: /content/RRDataset_subset.zip
100% 1.27G/1.27G [00:16<00:00, 75.5MB/s]


In [31]:
!rm -rf /content/data/raw/RRDataset_subset
!mkdir -p /content/data/raw
!unzip -n -q "$DATASET_ZIP_PATH" -d /content/data/raw

In [32]:
show_image_exists_check(train_df, IMAGE_ROOT, n=5)


Image root: /content/data/raw/RRDataset_subset
/content/data/raw/RRDataset_subset/original/real/real_006970.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_003543.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_004687.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_001434.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_004760.jpg OK


# Train

This cell launches training for the RGB multi-task baseline.


In [33]:
!python src/train.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint_name {CHECKPOINT_NAME} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --lambda_fake {LAMBDA_FAKE} \
  --lambda_transform {LAMBDA_TRANSFORM}


Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 163MB/s]

Epoch 1/10
Training:   0% 0/66 [00:02<?, ?it/s]
Traceback (most recent call last):
  File "/content/REPO/src/train.py", line 377, in <module>
    main()
  File "/content/REPO/src/train.py", line 313, in main
    train_metrics = train_one_epoch(
                    ^^^^^^^^^^^^^^^^
  File "/content/REPO/src/train.py", line 63, in train_one_epoch
    outputs = model(images)
              ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/REPO/

# Evaluation

This cell evaluates the best checkpoint and saves:

- `predictions.csv`;
- `metrics.json`;
- `confusion_fake.png`;
- `confusion_transform.png`.


In [34]:
!python src/evaluate.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint {CHECKPOINT_PATH} \
  --output_dir {OUTPUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}


Traceback (most recent call last):
  File "/content/REPO/src/evaluate.py", line 5, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/__init__.py", line 161, in <module>
    from . import _api, _version, cbook, _docstring, rcsetup
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/rcsetup.py", line 28, in <module>
    from matplotlib.colors import Colormap, is_color_like
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/colors.py", line 52, in <module>
    from PIL import Image
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 90, in <module>
    from . import _imaging as core
KeyboardInterrupt
^C


# Results

In [35]:
metrics_path = Path(OUTPUT_DIR) / 'metrics.json'

if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=4))
else:
    print(f'Metrics file not found: {metrics_path}')


Metrics file not found: results/rgb_baseline/metrics.json


In [36]:
predictions_path = Path(OUTPUT_DIR) / "predictions.csv"

if predictions_path.exists():
    predictions_df = pd.read_csv(predictions_path)
else:
    print(f"Predictions file not found: {predictions_path}")

Predictions file not found: results/rgb_baseline/predictions.csv


In [37]:
print("True fake labels:")
print(predictions_df["true_fake"].value_counts())

print("\nPredicted fake labels:")
print(predictions_df["pred_fake"].value_counts())

print("\nTrue transform labels:")
print(predictions_df["true_transform"].value_counts())

print("\nPredicted transform labels:")
print(predictions_df["pred_transform"].value_counts())

True fake labels:


NameError: name 'predictions_df' is not defined

In [ ]:
# Show saved confusion matrices if they exist.
for filename in ['confusion_fake.png', 'confusion_transform.png']:
    path = Path(OUTPUT_DIR) / filename
    if path.exists():
        img = plt.imread(path)
        plt.figure(figsize=(7, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(filename)
        plt.show()
    else:
        print(f'Missing: {path}')


#DEPTH

In [ ]:
!find /content -name "real_006970.jpg"

##Depth map generation

###first small debug

In [38]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small \
  --max_images 5

Using device: cuda
Number of images to process: 5
Loading pretrained MiDaS model: MiDaS_small
The repository intel-isl_MiDaS does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights:  None
The repository rwightman_gen-efficientnet-pytorch does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/z

###full depth map generation

In [39]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small

Using device: cuda
Number of images to process: 2550
Loading pretrained MiDaS model: MiDaS_small
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights:  None
Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Generating depth maps: 100% 2550/2550 [03:09<00:00, 13.46it/s]
Depth map generation completed.
Depth maps saved in: /content/drive/MyDrive/CV_Project/depth_maps


##copy of depth maph in local so colab doesn't need to go on drive every time

In [41]:
DRIVE_DEPTH_ROOT = "/content/drive/MyDrive/CV_Project/depth_maps"
LOCAL_DEPTH_ROOT = "/content/depth_maps"

!mkdir -p "{LOCAL_DEPTH_ROOT}"
!rsync -a --info=progress2 "{DRIVE_DEPTH_ROOT}/" "{LOCAL_DEPTH_ROOT}/"

DEPTH_ROOT = LOCAL_DEPTH_ROOT

print("Depth maps will be loaded from:", DEPTH_ROOT)

 13,665,777,036 100%  139.34MB/s    0:01:33 (xfr#2550, to-chk=0/2560)
Depth maps will be loaded from: /content/depth_maps


##First try with depth only

In [40]:
CHECKPOINT_NAME = "best_depth_only.pt"
!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name {CHECKPOINT_NAME} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --lambda_fake {LAMBDA_FAKE} \
  --lambda_transform {LAMBDA_TRANSFORM}

Using device: cuda

Epoch 1/10
Training geometric: 100% 66/66 [01:36<00:00,  1.47s/it]
Validation geometric: 100% 15/15 [00:24<00:00,  1.65s/it]
Train: {'loss': 1.4254548524674915, 'fake_acc': 0.8528571428571429, 'transform_acc': 0.42714285714285716}
Val:   {'loss': 1.087427446047465, 'fake_acc': 0.9088888888888889, 'transform_acc': 0.5711111111111111}
Val score: 0.7400
Learning rate: 0.000100
Saved best checkpoint to checkpoints/best_rgb.pt

Epoch 2/10
Training geometric: 100% 66/66 [01:37<00:00,  1.48s/it]
Validation geometric: 100% 15/15 [00:25<00:00,  1.69s/it]
Train: {'loss': 0.8633433499790373, 'fake_acc': 0.9657142857142857, 'transform_acc': 0.5966666666666667}
Val:   {'loss': 1.162972101105584, 'fake_acc': 0.8977777777777778, 'transform_acc': 0.6066666666666667}
Val score: 0.7522
Learning rate: 0.000100
Saved best checkpoint to checkpoints/best_rgb.pt

Epoch 3/10
Training geometric: 100% 66/66 [01:36<00:00,  1.45s/it]
Validation geometric: 100% 15/15 [00:25<00:00,  1.69s/it]
Tr

##Second try with also edge consistency

In [44]:
CHECKPOINT_NAME = "best_depth_full.pt"

!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name {CHECKPOINT_NAME} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --lambda_fake {LAMBDA_FAKE} \
  --lambda_transform {LAMBDA_TRANSFORM} \
  --no_edge

Using device: cuda

Epoch 1/10
Training geometric: 100% 66/66 [01:38<00:00,  1.49s/it]
Validation geometric: 100% 15/15 [00:25<00:00,  1.71s/it]
Train: {'loss': 1.4281242831548056, 'fake_acc': 0.8533333333333334, 'transform_acc': 0.41}
Val:   {'loss': 1.1005261222521463, 'fake_acc': 0.9088888888888889, 'transform_acc': 0.5688888888888889}
Val score: 0.7389
Learning rate: 0.000100
Saved best checkpoint to checkpoints/best_depth_full.pt

Epoch 2/10
Training geometric: 100% 66/66 [01:39<00:00,  1.51s/it]
Validation geometric: 100% 15/15 [00:25<00:00,  1.72s/it]
Train: {'loss': 0.8785466718673706, 'fake_acc': 0.9595238095238096, 'transform_acc': 0.5919047619047619}
Val:   {'loss': 1.0024488390816582, 'fake_acc': 0.9088888888888889, 'transform_acc': 0.6466666666666666}
Val score: 0.7778
Learning rate: 0.000100
Saved best checkpoint to checkpoints/best_depth_full.pt

Epoch 3/10
Training geometric: 100% 66/66 [01:36<00:00,  1.47s/it]
Validation geometric: 100% 15/15 [00:25<00:00,  1.73s/it]
T